In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
assert os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] == "false"

import sys
sys.path.append('../')

import ATLAS

import json, pickle
from glob import glob
from tqdm import tqdm

import numpy as np
from matplotlib import pyplot as plt

# Make Pint shut up
import pint.logging
import logging
pint.logging.setup(level=logging.WARNING)
logging.getLogger('enterprise').setLevel(logging.WARNING)

from pta_replicator.white_noise import add_measurement_noise, add_jitter
from pta_replicator.red_noise import add_red_noise, add_gwb
from pta_replicator.simulate import make_ideal, load_pulsar

NG15_path = '../datasets/NG15'
ideal_path = '../datasets/ideal_NG15'

JAX 64-bit mode automatically enabled successfully.


libstempo not installed. PINT or libstempo are required to use par and tim files.


In [2]:
def get_name(parOrTim):
    return parOrTim.split('/')[-1].split('_')[0]

def get_partim(path):
    parfiles = sorted(glob(f'{path}/par/*.par'))
    timfiles = sorted(glob(f'{path}/tim/*.tim'))
    assert len(parfiles) == len(timfiles)
    for p,t in zip(parfiles, timfiles):
        assert get_name(p) == get_name(t), f'{p, t}'
    return parfiles, timfiles

def plot_rep_psr(psr):
    t = psr.toas.get_mjds()
    r = psr.residuals.time_resids.to_value('s')
    e = psr.residuals.get_data_error().to_value('s')

    plt.errorbar(t, r, e, fmt='.', capsize=3, label=f'{psr.name}')
    plt.xlabel('MJD')
    plt.ylabel('Residuals (s)')
    plt.legend()

def make_dirs(path):
    if not os.path.exists(path):
        os.makedirs(path)
        os.makedirs(f'{path}/par')
        os.makedirs(f'{path}/tim')
    else:
        print(f'{path} already exists')
        import shutil
        shutil.rmtree(path)
        os.makedirs(path)
        os.makedirs(f'{path}/par')
        os.makedirs(f'{path}/tim')

def copy_partim(old_path, new_path, append_name=None):
    make_dirs(new_path)
    parfiles, timfiles = get_partim(old_path)
    for par, tim in zip(parfiles, timfiles):
        name = get_name(par)
        if append_name is not None:
            name += f'_{append_name}'

        os.system(f'cp {par} {new_path}/par/{name}.par')
        os.system(f'cp {tim} {new_path}/tim/{name}.tim')

def clean_par(path):
    parfiles, _ = get_partim(path)
    for par in parfiles:
        with open(par, 'r') as f:
            lines = f.readlines()
        to_remove = ['EFAC', 'EQUAD', 'ECORR', 'RNAMP', 'RNIDX']
        with open(par, 'w') as f:
            for line in lines:
                if not any(tr in line for tr in to_remove):
                    f.write(line)

def save_partim(psrs, path, name_append=None):
    for psr in tqdm(psrs, desc='Saving pulsars'):
        fname = f'{psr.name}_{name_append}' if name_append else f'{psr.name}'
        psr.write_partim(f'{path}/par/{fname}.par', f'{path}/tim/{fname}.tim', tempo2=False)

def get_backends(psr):
    return np.unique(psr.toas['f'])


## Idealize the NG15 dataset

I doubt you need to run this again

In [3]:
# Copy the par and tim files
copy_partim(NG15_path, ideal_path, append_name='ideal')

# Clean the par files
clean_par(ideal_path)

parfiles, timfiles = get_partim(ideal_path)
npsrs = len(parfiles)

rep_psrs = [load_pulsar(parfiles[i], timfiles[i]) for i in tqdm(range(npsrs), desc='Loading pulsars')]
[make_ideal(p) for p in tqdm(rep_psrs, desc='Idealizing')];

save_partim(rep_psrs, ideal_path, name_append='ideal')


../datasets/ideal_NG15 already exists


Saving pulsars: 100%|██████████| 67/67 [02:14<00:00,  2.00s/it]


## Making a simulated dataset

In [4]:
sim_path = '../datasets/sim_NG15_01'
copy_partim(ideal_path, sim_path, append_name='sim')

# No need to clean the par files, we already did that

# Load the pulsars
parfiles, timfiles = get_partim(sim_path)
npsrs = len(parfiles)
rep_psrs = [load_pulsar(parfiles[i], timfiles[i]) 
            for i in tqdm(range(npsrs), desc='Loading pulsars')]
[make_ideal(psr) for psr in tqdm(rep_psrs, desc='Making ideal')]

injected_params = {}
rng = np.random.default_rng(seed=420)
rseed = lambda : rng.integers(0, 2**32-1)

../datasets/sim_NG15_01 already exists


Loading pulsars:   4%|▍         | 3/67 [00:16<05:28,  5.13s/it]

Making ideal: 100%|██████████| 67/67 [05:28<00:00,  4.90s/it]


### White Noise

In [5]:

for psr in tqdm(rep_psrs, desc='Adding white noise'):
    backends = get_backends(psr)
    nbackends = len(backends)

    ef = rng.normal(loc=1.0, scale=0.2, size=nbackends)
    eq = rng.uniform(low=-8, high=-5, size=nbackends)
    ec = rng.uniform(low=-8, high=-5, size=nbackends)

    add_measurement_noise(psr, efac=ef, log10_equad=eq, flags=backends, seed=rseed())
    add_jitter(psr, log10_ecorr=ec, flags=backends, seed=rseed())
    
    injected_params.update({f'{psr.name}_{backends[i]}_efac': ef[i] for i in range(nbackends)})
    injected_params.update({f'{psr.name}_{backends[i]}_equad': eq[i] for i in range(nbackends)})
    injected_params.update({f'{psr.name}_{backends[i]}_ecorr': ec[i] for i in range(nbackends)})


Adding white noise: 100%|██████████| 67/67 [04:59<00:00,  4.47s/it]


### Intrinsic Red Noise

In [6]:
max_lkl_red ={
 'B1855+09': (3.3, -13.9), 'B1937+21': (3.7, -13.5), 'B1953+29': (2.9, -13.0),
 'J0023+0923': (1.2, -13.4), 'J0030+0451': (4.7, -14.4), 'J0340+4130': (2.6, -16.1),
 'J0406+3039': (1.5, -18.4), 'J0437-4715': (4.8, -17.1), 'J0509+0856': (0.51, -12.5),
 'J0557+1551': (5.6, -14.5), 'J0605+3757': (5.0, -14.2), 'J0610-2100': (1.6, -12.4),
 'J0613-0200': (2.6, -13.9), 'J0636+5128': (4.7, -19.8), 'J0645+5158': (0.25, -13.3),
 'J0709+0458': (2.3, -12.3), 'J0740+6620': (0.81, -14.3), 'J0931-1902': (2.6, -17.5),
 'J1012+5307': (0.046, -12.6), 'J1012-4235': (2.6, -17.4), 'J1022+1001': (3.5, -14.0),
 'J1024-0719': (2.2, -13.9), 'J1125+7819': (6.2, -15.1), 'J1312+0051': (1.4, -12.7),
 'J1453+1902': (2.9, -17.0), 'J1455-3330': (1.7, -15.3), 'J1600-3053': (3.1, -19.5),
 'J1614-2230': (0.9, -13.9), 'J1630+3734': (3.0, -16.7), 'J1640+2224': (6.8, -18.1),
 'J1643-1224': (1.5, -12.3), 'J1705-1903': (0.1, -12.2), 'J1713+0747': (0.8, -14.4),
 'J1719-1438': (4.1, -14.2), 'J1730-2304': (2.9, -13.1), 'J1738+0333': (5.7, -14.9),
 'J1741+1351': (2.0, -14.3), 'J1744-1134': (4.9, -17.3), 'J1745+1017': (2.0, -11.8),
 'J1747-4036': (3.0, -12.6), 'J1751-2857': (2.2, -19.3), 'J1802-2124': (0.2, -12.2),
 'J1811-2405': (4.0, -16.8), 'J1832-0836': (0.7, -19.7), 'J1843-1113': (1.7, -14.2),
 'J1853+1303': (1.1, -13.2), 'J1903+0327': (1.3, -12.1), 'J1909-3744': (5.4, -18.7),
 'J1910+1256': (0.87, -18.4), 'J1911+1347': (1.8, -14.2), 'J1918-0642': (6.1, -18.6),
 'J1923+2515': (2.8, -15.3), 'J1944+0907': (3.2, -13.7), 'J1946+3417': (1.0, -12.5),
 'J2010-1323': (0.03, -18.0), 'J2017+0603': (1.9, -17.0), 'J2033+1734': (5.1, -17.3),
 'J2043+1711': (2.5, -14.5), 'J2124-3358': (1.8, -16.7), 'J2145-0750': (0.5, -12.9),
 'J2214+3000': (3.6, -19.7), 'J2229+2643': (3.3, -17.9), 'J2234+0611': (4.5, -14.3),
 'J2234+0944': (4.8, -17.0), 'J2302+4442': (0.1, -18.1), 'J2317+1439': (3.7, -16.5),
 'J2322+2057': (3.5, -14.9),
}



In [7]:
for psr in tqdm(rep_psrs, desc='Adding intrinsic red noise'):
    gamma, log10_A = max_lkl_red[psr.name]
    add_red_noise(psr, log10_A, gamma, seed=rseed())
    injected_params.update({f'{psr.name}_irn_log10_A': log10_A, f'{psr.name}_irn_gamma': gamma})

Adding intrinsic red noise: 100%|██████████| 67/67 [02:32<00:00,  2.28s/it]


### Gravitational wave background

In [8]:
gwb_log10_A = np.log10(2.4e-15)
gwb_gamma = 13/3

add_gwb(rep_psrs, gwb_log10_A, gwb_gamma, seed=rseed())

injected_params.update({'gwb_log10_A': gwb_log10_A, 'gwb_gamma': gwb_gamma})

### Continuous waves 

If needed, this is where CW injections go

In [9]:
pass

### Outputing to the par and tim files

In [11]:
with open(f'{sim_path}/rep_psrs.pkl', 'wb') as f:
    pickle.dump(rep_psrs, f)

save_partim(rep_psrs, sim_path, 'sim')

with open(f'{sim_path}/injected_params.json', 'w') as f:
    json.dump(injected_params, f)

Saving pulsars: 100%|██████████| 67/67 [02:13<00:00,  1.99s/it]
